In [ ]:
import subprocess, sys
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', 'nnunetv2==2.8.1'], check=True)


In [ ]:
import os, json, shutil
from pathlib import Path
work = Path('/kaggle/working')
inputs = list(Path('/kaggle/input').rglob('prepare-report.json'))
if len(inputs) != 1:
    raise RuntimeError(f'Expected exactly one completed CPU preparation output, found {inputs}')
source = inputs[0].parent
report = json.loads(inputs[0].read_text())
if report['stage'] != 'cpu_preprocessing_complete' or report['labeled_cases'] != 484:
    raise RuntimeError('CPU preparation is incomplete')
shutil.copytree(source/'preprocessed', work/'preprocessed')
shutil.copy2(inputs[0], work/'prepare-report.json')
os.environ.update(nnUNet_raw=str(work/'raw'), nnUNet_preprocessed=str(work/'preprocessed'),
                  nnUNet_results=str(work/'results'), nnUNet_compile='false', nnUNet_n_proc_DA='2')
(work/'raw').mkdir(exist_ok=True)


In [ ]:
from pathlib import Path
Path('/kaggle/working/train_budget.py').write_text('"""100-epoch nnU-Net pilot with benchmark reporting and an epoch-boundary time budget.\n\nUses the upstream nnUNetTrainer_100epochs class so checkpoints remain compatible with\nordinary nnU-Net inference. A partial run is never labeled a completed 100-epoch model.\n"""\nimport argparse\nimport json\nimport os\nimport statistics\nimport time\nfrom pathlib import Path\n\n\nclass BudgetReached(Exception):\n    pass\n\n\ndef estimate(durations, completed, target):\n    steady = durations[2:] if len(durations) > 2 else durations\n    seconds = statistics.median(steady) if steady else None\n    return {\'seconds_per_epoch_median\': seconds,\n            \'estimated_remaining_hours\': (target-completed)*seconds/3600 if seconds else None}\n\n\ndef save_for_next_epoch(trainer, path):\n    """Save a boundary checkpoint with nnU-Net\'s next-epoch convention."""\n    next_epoch = trainer.current_epoch\n    trainer.current_epoch = next_epoch - 1\n    try:\n        trainer.save_checkpoint(str(path))\n    finally:\n        trainer.current_epoch = next_epoch\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument(\'--hours\', type=float, default=2.)\n    parser.add_argument(\'--fold\', type=int, default=0)\n    parser.add_argument(\'--resume\', type=Path)\n    args = parser.parse_args()\n    if not 0 < args.hours <= 8: parser.error(\'Budget must be in (0, 8] hours\')\n    if args.fold not in range(5): parser.error(\'Fold must be 0..4\')\n    os.environ.setdefault(\'nnUNet_compile\', \'false\')\n    os.environ.setdefault(\'nnUNet_n_proc_DA\', \'2\')\n    os.environ.setdefault(\'OMP_NUM_THREADS\', \'2\')\n    os.environ.setdefault(\'MKL_NUM_THREADS\', \'2\')\n    import torch\n    from importlib.metadata import version\n    from nnunetv2.run.run_training import get_trainer_from_args\n    if version(\'nnunetv2\') != \'2.8.1\': raise RuntimeError(\'Expected nnunetv2==2.8.1\')\n    if not torch.cuda.is_available(): raise RuntimeError(\'No GPU available; refusing CPU training\')\n    trainer = get_trainer_from_args(\'501\', \'2d\', args.fold,\n                                   trainer_name=\'nnUNetTrainer_100epochs\',\n                                   continue_training=bool(args.resume), device=torch.device(\'cuda\', 0))\n    if args.resume:\n        if not args.resume.is_file(): raise FileNotFoundError(args.resume)\n        trainer.load_checkpoint(str(args.resume))\n    trainer.save_every = 5\n    output = Path(trainer.output_folder)\n    output.mkdir(parents=True, exist_ok=True)\n    original_save = trainer.save_checkpoint\n    def atomic_save(filename):\n        target = Path(filename); temporary = target.with_suffix(target.suffix+\'.tmp\')\n        original_save(str(temporary))\n        temporary.replace(target)\n    trainer.save_checkpoint = atomic_save\n    started = time.monotonic()\n    durations = []\n    report_path = Path(\'/kaggle/working/train-report.json\')\n    def report(status):\n        result = {\'status\':status,\'target_epochs\':trainer.num_epochs,\'completed_epochs\':trainer.current_epoch,\n                  \'fold\':args.fold,\'configuration\':\'2d\',\'budget_hours\':args.hours,\n                  \'elapsed_hours\':(time.monotonic()-started)/3600,\n                  \'gpu\':torch.cuda.get_device_name(0),\'visible_gpu_count\':torch.cuda.device_count(),\n                  \'trainer\':trainer.__class__.__name__, \'full_volume_validation_performed\':False,\n                  \'metric_note\':\'Training log pseudo-Dice is not full-volume validation Dice.\',\n                  **estimate(durations,trainer.current_epoch,trainer.num_epochs)}\n        report_path.write_text(json.dumps(result,indent=2))\n        print(json.dumps(result),flush=True)\n    original_start, original_end = trainer.on_epoch_start, trainer.on_epoch_end\n    epoch_started = None\n    def start_epoch():\n        nonlocal epoch_started\n        elapsed = time.monotonic()-started\n        predicted = statistics.median(durations[-5:]) if durations else 0\n        if elapsed + predicted + 60 >= args.hours*3600:\n            raise BudgetReached()\n        epoch_started = time.monotonic()\n        original_start()\n    def end_epoch():\n        original_end()  # increments current_epoch\n        durations.append(time.monotonic()-epoch_started)\n        report(\'training\')\n    trainer.on_epoch_start, trainer.on_epoch_end = start_epoch, end_epoch\n    try:\n        trainer.run_training()\n        report(\'completed_100_epochs\')\n    except BudgetReached:\n        # save_checkpoint stores current_epoch+1; here current_epoch already\n        # points at the next epoch after on_epoch_end, so adjust temporarily.\n        if trainer.current_epoch > 0:\n            save_for_next_epoch(trainer, output/\'checkpoint_latest.pth\')\n        report(\'paused_budget_checkpoint_saved\' if trainer.current_epoch else \'paused_before_first_epoch\')\n    finally:\n        for name in (\'dataloader_train\',\'dataloader_val\'):\n            loader = getattr(trainer,name,None)\n            if hasattr(loader,\'_finish\'): loader._finish()\n    # Deliberately skip perform_actual_validation here: it is a separate GPU\n    # workload, to be budgeted explicitly after reviewing this pilot.\n\n\nif __name__ == \'__main__\': main()\n')
subprocess.run([sys.executable, '/kaggle/working/train_budget.py', '--hours', '2', '--fold', '0'], check=True)
shutil.rmtree('/kaggle/working/preprocessed')
print(Path('/kaggle/working/train-report.json').read_text())
